In [0]:
# Databricks notebook source

# COMMAND ----------
# Formula 1 Racing Data Platform
# Notebook: common\01_logger
#
# Purpose:
# Support log operations for the other notebooks.

In [0]:
# COMMAND ----------
# Imports

from datetime import datetime
from pyspark.sql import Row
from uuid import uuid4

In [0]:
# COMMAND ----------
# Create log table if it doesn't exists

def create_pipeline_execution_table():
    """
    Creates the metadata.pipeline_execution table if it does not exist.
    """

    spark.sql(f"""

    CREATE TABLE IF NOT EXISTS {CATALOG}.{METADATA_SCHEMA}.pipeline_execution
    (

        execution_id STRING,

        execution_timestamp TIMESTAMP,

        layer STRING,

        table_name STRING,

        rows_read BIGINT,

        rows_written BIGINT,

        rows_removed BIGINT,

        pct_rows_removed DOUBLE,

        execution_time_ms BIGINT,

        status STRING

    )

    USING DELTA

    """)

    print("✅ metadata.pipeline_execution is ready.")

In [0]:
# COMMAND ----------
# Logs pipeline execution

def log_pipeline_execution(
    layer: str,
    table_name: str,
    rows_read: int,
    rows_written: int,
    execution_time_ms: int,
    status: str = "SUCCESS"
):
    """
    Logs pipeline execution metrics into metadata.pipeline_execution.
    """

    rows_removed = rows_read - rows_written

    pct_rows_removed = (
        round((rows_removed / rows_read) * 100, 2)
        if rows_read > 0
        else 0
    )

    log_record = Row(

        execution_id=str(uuid4()),

        execution_timestamp=datetime.now(),

        layer=layer,

        table_name=table_name,

        rows_read=rows_read,

        rows_written=rows_written,

        rows_removed=rows_removed,

        pct_rows_removed=pct_rows_removed,

        execution_time_ms=execution_time_ms,

        status=status

    )

    log_df = spark.createDataFrame([log_record])

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            f"{CATALOG}.{METADATA_SCHEMA}.pipeline_execution"
        )
    )

    print(
        f"✅ Logged execution for {layer}.{table_name}"
    )